# PetBot: Atendimento Inicial do Pet Shop
## Projeto-modelo híbrido com Python, Pandas e Streamlit

Neste notebook construímos um chatbot de **atendimento inicial** para um pet shop fictício.

- **Camada 1 (regras):** palavras-chave, dicionários, funções e Pandas respondem às perguntas frequentes.
- **Camada 2 (LLM):** quando as regras não reconhecem a pergunta, o bot consulta um modelo de linguagem pela API da NVIDIA.



## 1. Preparando o ambiente

Se as bibliotecas ainda não estiverem instaladas, descomente e execute a linha abaixo no terminal ou em uma célula.

In [ ]:
# pip3 install pandas streamlit requests
import pandas as pd


## 2. Criando a base de dados

Cada linha é um **serviço** do pet shop. Os dados são fictícios e pedagógicos.

In [ ]:
import pandas as pd

servicos = [
    {"servico": "Banho", "categoria": "Higiene", "descricao": "Banho com shampoo e secagem", "porte_aceito": "Todos", "preco_a_partir": 60, "duracao_min": 60, "agendamento_necessario": "Sim", "observacao": "Traga a carteira de vacinação em dia e avise se o pet tem alergia ou fica agitado."},
    {"servico": "Banho e tosa", "categoria": "Higiene", "descricao": "Banho completo com tosa na máquina ou na tesoura", "porte_aceito": "Todos", "preco_a_partir": 120, "duracao_min": 120, "agendamento_necessario": "Sim", "observacao": "Diga ao agendar o tipo de tosa desejado (máquina ou tesoura). As vacinas precisam estar em dia."},
    {"servico": "Banho para gatos", "categoria": "Higiene", "descricao": "Banho especial para gatos com secagem cuidadosa", "porte_aceito": "Todos", "preco_a_partir": 70, "duracao_min": 60, "agendamento_necessario": "Sim", "observacao": "Traga o gato na caixa de transporte. Os gatos são atendidos em horário separado dos cães."},
    {"servico": "Tosa higiênica", "categoria": "Higiene", "descricao": "Tosa das patas barriga e região íntima", "porte_aceito": "Todos", "preco_a_partir": 70, "duracao_min": 60, "agendamento_necessario": "Sim", "observacao": "Pode ser feita junto com o banho, no mesmo horário."},
    {"servico": "Tosa para gatos", "categoria": "Higiene", "descricao": "Tosa higiênica adaptada para gatos", "porte_aceito": "Todos", "preco_a_partir": 80, "duracao_min": 60, "agendamento_necessario": "Sim", "observacao": "Traga o gato na caixa de transporte. Gatos muito agitados podem precisar de avaliação veterinária antes."},
    {"servico": "Hidratação de pelos", "categoria": "Higiene", "descricao": "Tratamento hidratante para pelos longos e ressecados", "porte_aceito": "Médio", "preco_a_partir": 90, "duracao_min": 60, "agendamento_necessario": "Sim", "observacao": "Indicada a cada 15 a 30 dias para pelos longos. Pode ser feita junto com o banho."},
    {"servico": "Corte de unhas", "categoria": "Higiene", "descricao": "Corte e lixamento das unhas", "porte_aceito": "Todos", "preco_a_partir": 25, "duracao_min": 15, "agendamento_necessario": "Não", "observacao": "Atendimento por ordem de chegada."},
    {"servico": "Escovação de dentes", "categoria": "Higiene", "descricao": "Limpeza dental com escova e pasta específica", "porte_aceito": "Todos", "preco_a_partir": 35, "duracao_min": 20, "agendamento_necessario": "Não", "observacao": "Em caso de tártaro, mau hálito forte ou gengiva sangrando, marque uma consulta veterinária."},
    {"servico": "Consulta veterinária", "categoria": "Saúde", "descricao": "Consulta clínica geral com veterinário parceiro", "porte_aceito": "Todos", "preco_a_partir": 150, "duracao_min": 40, "agendamento_necessario": "Sim", "observacao": "Traga a carteira de vacinação e exames anteriores. Em emergências, procure um pronto-socorro veterinário 24 horas."},
    {"servico": "Vacinação", "categoria": "Saúde", "descricao": "Aplicação de vacinas conforme o calendário", "porte_aceito": "Todos", "preco_a_partir": 120, "duracao_min": 20, "agendamento_necessario": "Sim", "observacao": "Traga a carteira de vacinação. O pet precisa estar saudável no dia; filhotes seguem o calendário indicado pelo veterinário."},
    {"servico": "Vermifugação", "categoria": "Saúde", "descricao": "Aplicação de vermífugo com orientação de dose", "porte_aceito": "Todos", "preco_a_partir": 40, "duracao_min": 10, "agendamento_necessario": "Não", "observacao": "A dose depende do peso, por isso o pet é pesado na hora."},
    {"servico": "Antipulgas", "categoria": "Saúde", "descricao": "Aplicação de antipulgas com orientação", "porte_aceito": "Todos", "preco_a_partir": 45, "duracao_min": 15, "agendamento_necessario": "Não", "observacao": "A dose depende do peso do pet. Repita no prazo indicado pelo produto."},
    {"servico": "Exame de sangue", "categoria": "Saúde", "descricao": "Coleta e envio para laboratório parceiro", "porte_aceito": "Todos", "preco_a_partir": 130, "duracao_min": 20, "agendamento_necessario": "Sim", "observacao": "O pet precisa de jejum de 8 horas de comida (água liberada). O resultado sai em até 2 dias úteis."},
    {"servico": "Hospedagem diária", "categoria": "Hospedagem", "descricao": "Hospedagem com área de descanso e alimentação", "porte_aceito": "Todos", "preco_a_partir": 80, "duracao_min": 1440, "agendamento_necessario": "Sim", "observacao": "Traga a ração do pet, a carteira de vacinação em dia e um objeto conhecido, como manta ou brinquedo."},
    {"servico": "Hospedagem para cães grandes", "categoria": "Hospedagem", "descricao": "Espaço amplo para cães de porte grande", "porte_aceito": "Grande", "preco_a_partir": 110, "duracao_min": 1440, "agendamento_necessario": "Sim", "observacao": "Traga a ração e a carteira de vacinação em dia. As vagas são limitadas, então reserve com antecedência."},
    {"servico": "Passeio", "categoria": "Hospedagem", "descricao": "Passeio de 30 minutos com monitor", "porte_aceito": "Todos", "preco_a_partir": 35, "duracao_min": 30, "agendamento_necessario": "Não", "observacao": "O pet precisa estar com as vacinas em dia. O passeio é feito com coleira e guia."},
    {"servico": "Adestramento básico", "categoria": "Adestramento", "descricao": "Sessão de comandos básicos como sentar e deitar", "porte_aceito": "Todos", "preco_a_partir": 100, "duracao_min": 45, "agendamento_necessario": "Sim", "observacao": "Os melhores resultados aparecem com sessões semanais e treino em casa."},
    {"servico": "Adestramento de filhotes", "categoria": "Adestramento", "descricao": "Socialização e educação para filhotes", "porte_aceito": "Pequeno", "preco_a_partir": 90, "duracao_min": 45, "agendamento_necessario": "Sim", "observacao": "Indicado a partir de 3 meses de idade, depois das primeiras vacinas."},
    {"servico": "Transporte pet", "categoria": "Transporte", "descricao": "Busca e entrega do pet na casa do cliente", "porte_aceito": "Todos", "preco_a_partir": 30, "duracao_min": 40, "agendamento_necessario": "Sim", "observacao": "Atende bairros próximos à loja. Cães pequenos e gatos viajam na caixa de transporte."},
    {"servico": "Ração premium", "categoria": "Alimentação", "descricao": "Ração premium para cães e gatos vendida por pacote", "porte_aceito": "Todos", "preco_a_partir": 120, "duracao_min": 0, "agendamento_necessario": "Não", "observacao": "Temos opções para filhotes, adultos e idosos. Consulte as marcas disponíveis na loja."},
    {"servico": "Petisco natural", "categoria": "Alimentação", "descricao": "Petiscos naturais sem conservantes", "porte_aceito": "Todos", "preco_a_partir": 15, "duracao_min": 0, "agendamento_necessario": "Não", "observacao": "Ofereça com moderação, sem substituir a refeição do pet."},
]

df = pd.DataFrame(servicos)
# index=False não traz o índice; encoding utf-8 mantém os acentos do português
df.to_csv("dados.csv", index=False, encoding="utf-8")
df

## 3. Investigação da base com Pandas

Antes de programar as respostas, conhecemos os dados.

In [ ]:
df = pd.read_csv("dados.csv")

print("Dimensão da base:", df.shape)
print("Colunas:", list(df.columns))
print("\nValores ausentes:")
print(df.isnull().sum())
print("\nQuantidade por categoria:")
print(df["categoria"].value_counts())
print("\nServiços com agendamento:")
print(df[df["agendamento_necessario"] == "Sim"]["servico"].tolist())

## 4. Dicionários de intenções e informações da loja

Uma **intenção** é o que o cliente deseja saber sobre um serviço. A ordem importa: intenções mais específicas vêm primeiro.

Também guardamos as informações da loja (horário, pagamento, endereço e contato), que são as perguntas mais comuns do dia a dia. Os dados são fictícios.

In [ ]:
intencoes = {
    "agendamento": ["agendar", "marcar", "agendamento"],
    "preco": ["quanto custa", "preço", "preco", "valor"],
    "duracao": ["quanto tempo", "demora", "duração", "duracao"],
    "descricao": ["o que inclui", "como funciona", "o que é"],
    "porte": ["porte", "tamanho"],
    "listar": ["liste", "listar", "quais serviços", "quais servicos", "mostre"]
}

# informações da loja (fictícias), as perguntas mais comuns do dia a dia
WHATSAPP = "(11) 90000-0000"

informacoes_loja = {
    "horario": "Funcionamos de segunda a sexta, das 8h às 19h, e aos sábados, das 8h às 14h. Fechamos aos domingos e feriados.",
    "pagamento": "Aceitamos Pix, dinheiro e cartões de débito e crédito. Serviços acima de R$ 200,00 podem ser parcelados em até 3 vezes sem juros no crédito.",
    "endereco": "Estamos na Rua das Flores, 123, Centro, com estacionamento gratuito para clientes. Se preferir, o serviço Transporte pet busca e entrega o seu pet em casa.",
    "contato": "Fale com a equipe pelo WhatsApp " + WHATSAPP + " ou pelo telefone (11) 3000-0000, no horário de funcionamento."
}

palavras_loja = {
    "horario": ["horário", "horario", "que horas", "abre", "fecha", "funcionamento", "domingo", "sábado", "sabado", "feriado"],
    "pagamento": ["pagamento", "pagar", "pix", "cartão", "cartao", "dinheiro", "parcela", "crédito", "credito", "débito", "debito"],
    "endereco": ["endereço", "endereco", "onde fica", "localização", "localizacao", "estacionamento"],
    "contato": ["telefone", "whatsapp", "contato", "falar com"]
}

## 5. Funções para identificar a intenção e o assunto da loja

In [ ]:
def descobrir_intencao(pergunta):
    texto = pergunta.lower()
    for intencao, palavras in intencoes.items():
        for palavra in palavras:
            if palavra in texto:
                return intencao
    return "desconhecida"

def descobrir_assunto_loja(pergunta):
    texto = pergunta.lower()
    for assunto, palavras in palavras_loja.items():
        for palavra in palavras:
            if palavra in texto:
                return assunto
    return None

perguntas_teste = [
    "Quanto custa um banho?",
    "Como agendar a vacinação?",
    "Quanto tempo leva a tosa?",
    "Qual a previsão do tempo?"
]

for pergunta in perguntas_teste:
    print(pergunta, "->", descobrir_intencao(pergunta))

print("Vocês aceitam Pix? ->", descobrir_assunto_loja("Vocês aceitam Pix?"))

## 6. Identificando o serviço mencionado

Se dois nomes aparecem na pergunta (por exemplo, "Banho" e "Banho e tosa"), escolhemos o **mais longo**.

In [ ]:
def descobrir_servico(pergunta, dataframe):
    texto = pergunta.lower()
    encontrados = [servico for servico in dataframe["servico"] if servico.lower() in texto]
    if not encontrados:
        return None
    return max(encontrados, key=len)  # o nome mais longo vence: "Banho e tosa" vence "Banho"

print(descobrir_servico("Quanto tempo leva o banho e tosa?", df))
print(descobrir_servico("Quanto custa um banho?", df))

## 7. Consultas com Pandas e montagem das respostas

Além do filtro no DataFrame, criamos funções pequenas para formatar o preço (R$ 60,00), a duração (2 horas), explicar como agendar e montar um resumo completo do serviço.

In [ ]:
def buscar_servico(servico, dataframe):
    resultado = dataframe[dataframe["servico"].str.lower() == servico.lower()]
    if len(resultado) == 0:
        return None
    return resultado.iloc[0]

def formatar_preco(valor):
    return f"R$ {float(valor):.2f}".replace(".", ",")  # 60 vira "R$ 60,00"

def formatar_duracao(minutos):
    if minutos < 60:
        return f"{minutos} minutos"
    horas = minutos // 60
    if horas == 1:
        return "1 hora"
    return f"{horas} horas"

def como_agendar(item):
    if item["agendamento_necessario"] == "Sim":
        return (f"{item['servico']} precisa de agendamento: chame no WhatsApp {WHATSAPP} "
                "informando o nome, a raça e o porte do pet, ou agende direto na loja.")
    return f"{item['servico']} não precisa de agendamento: é só vir à loja no horário de funcionamento."

def resumo_servico(item):
    minutos = int(item["duracao_min"])
    texto = f"{item['servico']}: {item['descricao']}. Preço a partir de {formatar_preco(item['preco_a_partir'])}"
    if minutos > 0:
        texto = texto + f", com duração de cerca de {formatar_duracao(minutos)}"
    return texto + f". {item['observacao']} {como_agendar(item)}"

item = buscar_servico("Vacinação", df)
print(formatar_preco(item["preco_a_partir"]))
print(resumo_servico(item))

## 8. Camada LLM: consulta à API da NVIDIA

A API da NVIDIA é compatível com o formato da OpenAI. Usamos `requests` para enviar a pergunta.

A chave **não** fica no código. Ela é lida da variável de ambiente `NVIDIA_API_KEY`, criada em build.nvidia.com.
O prompt inclui o catálogo como texto, para que o modelo responda com os dados da base.

Modelo usado: `nvidia/nemotron-3.5-bolt-e4b` (leve e rápido). Se a API falhar, a mensagem mostra o código HTTP para facilitar o diagnóstico. Modelos com raciocínio podem devolver o texto em `reasoning_content`; a função também trata esse caso.

In [ ]:
import os
import requests

URL_LLM = "https://integrate.api.nvidia.com/v1/chat/completions"
MODELO_LLM = "nvidia/nemotron-3.5-bolt-e4b"  # confirme o modelo disponível em build.nvidia.com

def perguntar_llm(pergunta, dataframe):
    chave = os.getenv("NVIDIA_API_KEY")
    if not chave:
        return "O assistente avançado não está configurado. Fale com a equipe pelo WhatsApp " + WHATSAPP + "."
    catalogo = dataframe.to_string(index=False)
    loja = "\n".join(informacoes_loja.values())
    instrucao = ("Você é o atendente virtual de um pet shop. Responda em português, de forma clara e prática, "
                 "em até 4 frases, somente sobre o pet shop e os cuidados com pets. "
                 "Use apenas as informações abaixo e não invente preços, serviços ou horários. "
                 "Quando indicar um serviço, informe o preço inicial e se precisa de agendamento. "
                 "Termine com o próximo passo para o cliente, como agendar pelo WhatsApp ou vir à loja. "
                 "Se a informação não estiver abaixo, diga que não sabe e indique o contato da loja. "
                 "Em caso de emergência ou sintoma grave, oriente a procurar um veterinário imediatamente.\n\n"
                 "Informações da loja:\n" + loja + "\n\nCatálogo de serviços:\n" + catalogo)
    dados = {
        "model": MODELO_LLM,
        "messages": [{"role": "system", "content": instrucao},
                     {"role": "user", "content": pergunta}],
        "max_tokens": 2048,
        "temperature": 0.2
    }
    try:
        resposta = requests.post(URL_LLM, headers={"Authorization": "Bearer " + chave},
                                 json=dados, timeout=60)
        if resposta.status_code != 200:
            return f"No momento não consigo responder (erro {resposta.status_code}). Fale com a equipe pelo WhatsApp {WHATSAPP}."
        mensagem = resposta.json()["choices"][0]["message"]
        # alguns modelos colocam o texto em "reasoning_content" e deixam "content" vazio
        texto = mensagem.get("content") or mensagem.get("reasoning_content") or ""
        if texto == "":
            return "No momento não consigo responder essa pergunta. Fale com a equipe pelo WhatsApp " + WHATSAPP + "."
        return texto
    except (requests.RequestException, KeyError, IndexError):
        return "No momento não consigo responder essa pergunta. Fale com a equipe pelo WhatsApp " + WHATSAPP + "."

## 9. O cérebro do PetBot

O `chatbot` combina as etapas: emergência, saudação, intenção, serviço, consulta com Pandas, resposta e, se necessário, o LLM.

In [ ]:
palavras_emergencia = ["sangrando", "sangramento", "sangrou", "convulsão", "convulsao", "envenenado", "envenenou",
                       "desmaiou", "não respira", "nao respira", "atropelado", "engasgou", "engasgado"]

def chatbot(pergunta, dataframe, usar_llm=True):
    texto = pergunta.lower().strip()
    if not texto:
        return "Digite uma pergunta para que eu possa ajudar."

    if any(palavra in texto for palavra in palavras_emergencia):
        return ("Atenção: isso pode ser uma emergência. Leve o pet a um pronto-socorro veterinário 24 horas imediatamente. "
                "Enquanto isso, mantenha o pet calmo e aquecido, não dê remédios por conta própria e, se houver "
                "sangramento, pressione o local com um pano limpo. Se precisar, ligue para nós: " + WHATSAPP + ".")

    saudacoes = ["oi", "olá", "ola", "bom dia", "boa tarde", "boa noite"]
    if any(s == texto or texto.startswith(s + " ") for s in saudacoes):
        return ("Olá! Eu sou o PetBot, o atendente virtual do pet shop. Posso informar preço, duração, "
                "agendamento e cuidados de cada serviço, além do horário de funcionamento, das formas de "
                "pagamento, do endereço e do contato. Como posso ajudar?")

    frase = texto.replace("?", "").strip()  # a frase precisa ser exatamente uma destas
    if frase in ["ajuda", "preciso de ajuda", "o que você faz", "o que voce faz"]:
        return ("Você pode perguntar, por exemplo: 'Quanto custa um banho?', 'Como agendar a vacinação?', "
                "'O que inclui o exame de sangue?', 'Liste os serviços de Saúde', "
                "'Qual o horário de funcionamento?' ou 'Vocês aceitam Pix?'.")

    intencao = descobrir_intencao(pergunta)

    if intencao == "listar":
        for categoria in dataframe["categoria"].dropna().unique():
            if categoria.lower() in texto:
                encontrados = dataframe[dataframe["categoria"].str.lower() == categoria.lower()]
                itens = []
                for _, linha in encontrados.iterrows():
                    itens.append(f"{linha['servico']} (a partir de {formatar_preco(linha['preco_a_partir'])})")
                return f"Serviços de {categoria}: " + ", ".join(itens) + ". Pergunte sobre qualquer um deles para saber mais."
        partes = []
        for categoria in dataframe["categoria"].unique():
            nomes = dataframe[dataframe["categoria"] == categoria]["servico"].tolist()
            partes.append(f"{categoria}: " + ", ".join(nomes))
        return "Nossos serviços por categoria. " + ". ".join(partes) + ". Para ver os preços, peça, por exemplo, 'Liste os serviços de Higiene'."

    servico = descobrir_servico(pergunta, dataframe)
    if servico is None or intencao == "desconhecida":
        assunto = descobrir_assunto_loja(pergunta)
        if assunto is not None:
            return informacoes_loja[assunto]
        if usar_llm:
            return perguntar_llm(pergunta, dataframe)
        if servico is not None:
            return resumo_servico(buscar_servico(servico, dataframe))
        return ("Ainda não entendi. Pergunte sobre preço, duração, agendamento, descrição ou porte de um serviço, "
                "ou sobre horário, pagamento, endereço e contato. Se preferir, fale com a equipe pelo WhatsApp " + WHATSAPP + ".")

    item = buscar_servico(servico, dataframe)
    minutos = int(item["duracao_min"])
    if intencao == "preco":
        return (f"{servico} custa a partir de {formatar_preco(item['preco_a_partir'])}. O valor final é confirmado "
                f"no atendimento, conforme o porte e as condições do pet. {como_agendar(item)}")
    elif intencao == "duracao":
        if minutos == 0:
            return f"{servico} é um produto: pode ser comprado direto na loja, sem espera. {item['observacao']}"
        return f"{servico} leva cerca de {formatar_duracao(minutos)}. {item['observacao']} {como_agendar(item)}"
    elif intencao == "agendamento":
        return f"{como_agendar(item)} {item['observacao']}"
    elif intencao == "descricao":
        return resumo_servico(item)
    elif intencao == "porte":
        if item["porte_aceito"] == "Todos":
            return f"{servico} atende pets de todos os portes. {item['observacao']}"
        return f"{servico} atende pets de porte {item['porte_aceito'].lower()}. {item['observacao']}"
    return resumo_servico(item)

## 10. Testando o chatbot no notebook

Primeiro testamos só as regras (`usar_llm=False`), que não dependem de internet.

In [ ]:
testes = [
    "Oi",
    "Quanto custa um banho?",
    "Quanto tempo leva o banho e tosa?",
    "Como agendar a vacinação?",
    "O que inclui o exame de sangue?",
    "Qual o porte aceito pela hidratação de pelos?",
    "Liste os serviços de Saúde",
    "Qual o horário de funcionamento?",
    "Vocês aceitam Pix?",
    "Tenho um cão que está sangrando"
]

for pergunta in testes:
    print("Cliente:", pergunta)
    print("PetBot:", chatbot(pergunta, df, usar_llm=False))
    print("-" * 60)

## 11. Testando a camada LLM

Esta célula só chama a API se a variável `NVIDIA_API_KEY` estiver definida. Sem a chave, o bot informa que o assistente não está configurado.

A pergunta abaixo não é reconhecida pelas regras, por isso vai para o LLM.

In [ ]:
print("PetBot:", chatbot("Meu cachorro é idoso, qual serviço vocês recomendam?", df, usar_llm=True))